# The model-selection budget — structural variance decomposition (capstone)

**NOT FOR CLINICAL USE.** Population/trial-level variance accounting only; no individual prediction, no therapy ranking, no model recommendation. Executed in CI (nbmake).

Onkos has, one axis at a time, made every structural choice in a composed survival forecast first-class (which TGI model, whether parameters are estimable, which interaction model, which resistance mechanism, which survival metric/structure). This notebook accounts for them **together**: a balanced two-way variance-component decomposition (ANOVA / first-order Sobol over the structural factors) that splits the total forecast variance into parameter noise vs the variance contributed by each structural choice — and names which assumption to standardize first. See `docs/specs/research/model-selection-budget.md`.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import onkos
from onkos.budget import model_selection_budget, variance_components

ds = onkos.load()
print(f'{len(ds)} records | onkos {onkos.__version__}')

## The decomposition

`Var(Q) = WITHIN(parameter) + V_model(TGI choice) + V_link(survival choice) + V_inter`. The two structural factors are the in-context TGI models (`compare().included`) and every eligible survival link (week-8 Weibull, Cox, k_g). Each (model, link) cell runs the parameter-IIV ensemble; the variance components are the balanced two-way ANOVA split.

In [ ]:
ctx = {'tumor_type': 'NSCLC', 'line': 'first'}
b = model_selection_budget(ds, context=ctx, endpoint='OS', n=200)
print('TGI models :', [m.split('.')[1] for m in b.models])
print('survival links:', [l.split('.')[-1] for l in b.links])
print(f'grand-mean median OS = {b.grand_mean:.1f} wk | total variance = {b.total:.1f} | tier {b.tier}\n')
for axis, frac in sorted(b.fractions.items(), key=lambda kv: kv[1], reverse=True):
    print(f'  {axis:<14} {frac*100:5.1f}%')
print(f'\nstructural share = {b.structural_fraction*100:.0f}% (irreducible by a bigger trial)')
print(f'dominant axis    = {b.dominant} -> standardize / validate this first')

## The capstone finding

For NSCLC OS the **largest single component is the model×link interaction** — the v0.25 result that the survival metric can *invert* which TGI model wins is exactly an interaction term, and here it dominates. Most of the forecast uncertainty is irreducible structural-choice risk, not parameter noise a bigger trial would shrink, and the survival-model axis matters more than the tumor-growth model everyone argues about.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 1.8))
order = ['parameter', 'tgi_model', 'survival_link', 'interaction']
colors = {'parameter': '#3182ce', 'tgi_model': '#b7791f', 'survival_link': '#2f855a', 'interaction': '#c53030'}
left = 0.0
for k in order:
    f = b.fractions[k]
    ax.barh([0], [f], left=left, color=colors[k], label=k)
    if f > 0.04:
        ax.text(left + f/2, 0, f'{f*100:.0f}%', ha='center', va='center', color='white', fontweight='bold')
    left += f
ax.set(xlim=(0, 1), yticks=[], xlabel='share of total forecast variance', title='NSCLC 1L OS — model-selection budget')
ax.legend(fontsize=7, ncol=4, loc='upper center', bbox_to_anchor=(0.5, -0.4))
plt.show()

## Collapse to v0.21, and the closed-form guarantees

Collapsing the survival-link factor to one level recovers exactly the v0.21 within/between split — the budget is a strict generalization. The variance-component algebra is landmark-tested (sum identity, non-negativity, single-factor collapse, additive→zero-interaction, ANOVA-SS match).

In [ ]:
# Single-link collapse == v0.21 within/between (V_link = V_inter = 0).
means = np.array([[40.0], [70.0], [95.0]]); within = np.array([[2.0], [3.0], [4.0]])
c = variance_components(means, within)
assert np.isclose(c['v_link'], 0) and np.isclose(c['v_inter'], 0)
assert np.isclose(c['v_model'], c['between']) and np.isclose(c['between'], np.var([40, 70, 95]))

# Components sum to total; fractions to 1; all non-negative.
assert np.isclose(c['within'] + c['v_model'] + c['v_link'] + c['v_inter'], c['total'])
d = b.to_dict()
assert d['NOT_FOR_CLINICAL_USE'] is True and 'PROHIBITED' in d['onkos:clinicalUse']
print('v0.21 collapse + guarantees OK | structural fraction in dict:', round(d['structural_fraction'], 3))

## Curation triage: which contexts are structure-dominated

`onkos report` ranks contexts by whether their forecast uncertainty is dominated by structural choice (standardize an assumption) or parameter noise (run a bigger trial), and flags contexts with only one survival link — where the survival-model axis is not even cross-checked.

In [ ]:
for tt, ln in [('NSCLC','first'),('CRC','first'),('HCC','first'),('melanoma','first'),('breast','first'),('NSCLC','second')]:
    bb = model_selection_budget(ds, context={'tumor_type': tt, 'line': ln}, endpoint='OS', n=120)
    dom = 'structure' if bb.structural_fraction >= 0.5 else 'parameter'
    print(f'{tt:9} {ln:7} {len(bb.models)}x{len(bb.links)} grid  structural {bb.structural_fraction*100:4.0f}%  -> {dom}-dominated')